# 07_session_id — configure the JWT session claim and read the session id

A JWT authenticated by `JwtAuthCoordinator` identifies the caller (`sub`) and the login session (here the `sid` claim). `session_id_claim` is a required constructor argument. An aspect reads the session id like any other context field: `@context_requires(Ctx.User.session_id)` and `ctx.get(Ctx.User.session_id)`.

The token issuer owns the session lifecycle. Here the service issues its own tokens: every login gets a new session id, and a refresh keeps it.

Script version: `examples/step_13_fastapi/07_session_id.py`. Guide: `docs/extensions/jwt.md`, section "The session id".

In [ ]:
!pip install "aoa-action-machine[jwt]" aoa-fastapi-adapter

In [1]:
from datetime import UTC, datetime, timedelta
from itertools import count
from uuid import uuid4
import jwt
from fastapi.testclient import TestClient
from pydantic import Field
from aoa.action_machine.auth import ApplicationRole, GuestRole, NoAuthCoordinator
from aoa.action_machine.auth.jwt_auth import JwtAuthCoordinator
from aoa.action_machine.context import Context, ContextView, Ctx
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.context_requires import context_requires
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult, ParamsStub
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox
from aoa.fastapi import FastApiAdapter

## The session claim name is required

The coordinator is refused at construction when `session_id_claim` is left out (`TypeError`) or is not a non-empty string (`ValueError`). There is no default and no way to switch the check off.

In [2]:
_SECRET_KEY = "demo-only-secret-do-not-use-in-production"

_ALGORITHM = "HS256"

_TOKEN_TTL = timedelta(minutes=30)

_session_numbers = count(1)


class AuthDomain(BaseDomain):
    name = "auth"
    description = "Login, refresh and identity"

class ViewerRole(ApplicationRole):
    name = "viewer"
    description = "Read-only access."

In [3]:
def show_configuration_rules() -> None:
    """The session claim name is required and must be a non-empty string."""
    try:
        JwtAuthCoordinator(secret_key=_SECRET_KEY, role_registry={"viewer": ViewerRole})
        raise AssertionError("a coordinator without session_id_claim must not be built")
    except TypeError as exc:
        print(f"no session_id_claim  -> {type(exc).__name__}: {exc}")
        assert "session_id_claim" in str(exc)

    try:
        JwtAuthCoordinator(secret_key=_SECRET_KEY, role_registry={"viewer": ViewerRole}, session_id_claim="")
        raise AssertionError("an empty session_id_claim must be refused")
    except ValueError as exc:
        print(f"session_id_claim=''  -> {type(exc).__name__}: {exc}")
        assert str(exc) == "session_id_claim must be a non-empty string; got ''."


show_configuration_rules()

no session_id_claim  -> TypeError: JwtAuthCoordinator.__init__() missing 1 required keyword-only argument: 'session_id_claim'
session_id_claim=''  -> ValueError: session_id_claim must be a non-empty string; got ''.


## The issuer and the Actions

`_sign` puts three identifiers into a token: `sub` (the caller), `sid` (the login session) and `jti` (this one token). `LoginAction` starts a new session; `RefreshAction` signs a new token for the session it reads from the context; `WhoAmIAction` reports both identifiers.

In [4]:
def _sign(user_id: str, session_id: str) -> str:
    """Sign a token for one caller in one login session; ``jti`` identifies the token itself."""
    now = datetime.now(UTC)
    payload = {"sub": user_id, "sid": session_id, "jti": uuid4().hex, "roles": ["viewer"], "iat": now, "exp": now + _TOKEN_TTL}
    return jwt.encode(payload, _SECRET_KEY, algorithm=_ALGORITHM)

class LoginParams(BaseParams):
    username: str = Field(description="Account username")

class TokenResult(BaseResult):
    access_token: str = Field(description="Signed JWT bearer token")

class IdentityResult(BaseResult):
    user_id: str = Field(description="Caller identifier from the token")
    session_id: str = Field(description="Login session identifier from the token")

In [5]:
@meta(description="Start a new login session and issue its first token.", domain=AuthDomain)
@check_roles(GuestRole)
class LoginAction(BaseAction[LoginParams, TokenResult]):

    @summary_aspect("Sign a token with a new session id.")
    async def login_summary(self, params: LoginParams, state: BaseState, box: ToolsBox, connections: Connections) -> TokenResult:
        session_id = f"session-{next(_session_numbers)}"
        return TokenResult(access_token=_sign(params.username, session_id))

@meta(description="Issue a fresh token for the current login session.", domain=AuthDomain)
@check_roles(ViewerRole)
class RefreshAction(BaseAction[ParamsStub, TokenResult]):

    @summary_aspect("Sign a fresh token that keeps the session id.")
    @context_requires(Ctx.User.user_id, Ctx.User.session_id)
    async def refresh_summary(self, params: ParamsStub, state: BaseState, box: ToolsBox, connections: Connections, ctx: ContextView) -> TokenResult:
        return TokenResult(access_token=_sign(ctx.get(Ctx.User.user_id), ctx.get(Ctx.User.session_id)))

@meta(description="Report who is calling and from which session.", domain=AuthDomain)
@check_roles(ViewerRole)
class WhoAmIAction(BaseAction[ParamsStub, IdentityResult]):

    @summary_aspect("Read the caller and the session from the context.")
    @context_requires(Ctx.User.user_id, Ctx.User.session_id)
    async def whoami_summary(self, params: ParamsStub, state: BaseState, box: ToolsBox, connections: Connections, ctx: ContextView) -> IdentityResult:
        return IdentityResult(user_id=ctx.get(Ctx.User.user_id), session_id=ctx.get(Ctx.User.session_id))

## The application

`/auth/login` has no token to present yet, so it opts out of the JWT default with `NoAuthCoordinator`. The other two routes require a valid token with a `sid` claim.

In [6]:
def build_app():
    auth = JwtAuthCoordinator(
        secret_key=_SECRET_KEY,
        algorithm=_ALGORITHM,
        role_registry={"viewer": ViewerRole},
        session_id_claim="sid",
    )
    return (
        FastApiAdapter(machine=ActionProductMachine(), auth_coordinator=auth, title="Session Id Demo")
        # Login has no token to present yet, so it opts out of the JWT default.
        .post("/auth/login", LoginAction, auth_coordinator=NoAuthCoordinator(context=Context()))
        .post("/auth/refresh", RefreshAction)
        .get("/me", WhoAmIAction)
        .build()
    )


client = TestClient(build_app(), raise_server_exceptions=False)


def login(username: str) -> str:
    return client.post("/auth/login", json={"username": username}).json()["access_token"]


def whoami(token: str) -> dict[str, str]:
    response = client.get("/me", headers={"Authorization": f"Bearer {token}"})
    assert response.status_code == 200, response.text
    return response.json()

## Two logins are two sessions

The same user logs in twice. Both tokens carry `sub="alice"`, but each login got its own session id, and the aspect sees the difference.

In [7]:
laptop = login("alice")
phone = login("alice")
print(f"GET /me (laptop token)    -> {whoami(laptop)}")
print(f"GET /me (phone token)     -> {whoami(phone)}")
assert whoami(laptop) == {"user_id": "alice", "session_id": "session-1"}
assert whoami(phone) == {"user_id": "alice", "session_id": "session-2"}

GET /me (laptop token)    -> {'user_id': 'alice', 'session_id': 'session-1'}
GET /me (phone token)     -> {'user_id': 'alice', 'session_id': 'session-2'}


## A refresh keeps the session

`RefreshAction` reads the session id from the context and signs a new token with it. The token is new; the session is the same.

In [8]:
refreshed = client.post("/auth/refresh", headers={"Authorization": f"Bearer {laptop}"}).json()["access_token"]
print(f"refreshed token is new    -> {refreshed != laptop}")
print(f"GET /me (refreshed token) -> {whoami(refreshed)}")
assert refreshed != laptop
assert whoami(refreshed) == whoami(laptop)

refreshed token is new    -> True
GET /me (refreshed token) -> {'user_id': 'alice', 'session_id': 'session-1'}
